# Access Beyond the Map
### Reported barriers to essential services · reproducible Colab project

[Open in Colab](https://colab.research.google.com/github/zen0072/fall2026_datathon/blob/main/Access_Beyond_the_Map.ipynb)

**Question:** Which essential-service locations have review evidence of physical, sensory/communication, language, financial, or time barriers worth verifying?

Maps show where services are. This project combines **Places + Reviews** to reveal reported barriers to using them. It supports manual verification, not accessibility certification, a legal finding, or a claim that a new facility is necessary.

**Run:** Runtime → Run all. The default `synthetic` mode tests every stage without credentials. For actual analysis, set `MODE = "real"`, place the two full organizer CSVs in `DATA_DIR`, and rerun from a fresh runtime. Optional adjudicated annotations go in `labels.csv`. No paid APIs or public deployment.

**Execution status:** All sections contain implementation and explanations. Real source files and human labels were not accessible during authoring. Synthetic scores are software checks, never datathon findings. In real mode without gold labels, the trained model uses explicitly provisional rule labels and independent evaluation stays blocked. A video script is exported; recording remains outstanding.

**Sections:** 1 setup; 2 data acquisition; 3 audit/join; 4 scope/deduplication/splits; 5 labeling; 6 models; 7 held-out evaluation; 8 error analysis; 9 aggregation; 10 interactive map/list; 11 integrity checks; 12 reports/slides/video script; 13 reproducibility/export.


## 1 · Reproducible environment
CPU is sufficient. Package versions below target Python 3.11–3.12. Outputs remain in the runtime under `outputs/`; download the ZIP at the end before disconnecting. Never commit raw data, annotations, models trained on private text, or executed real-data notebook outputs to this public repository without checking redistribution rights.


In [ ]:
%pip -q install numpy==2.0.2 pandas==2.2.3 scikit-learn==1.6.1 scipy==1.14.1 matplotlib==3.9.4 folium==0.19.5 ipywidgets==8.1.5 datasketch==1.6.5 joblib==1.4.2


In [ ]:
from pathlib import Path
import re, json, hashlib, html, sys, platform, shutil, warnings
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, clear_output
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, f1_score, confusion_matrix, cohen_kappa_score
from datasketch import MinHash, MinHashLSH
import joblib, folium, ipywidgets as widgets

SEED = 42
MODE = "synthetic"  # @param ["synthetic", "real"]
DATA_DIR = Path("/content/data" if Path("/content").exists() else "data")
OUT = Path("outputs") / MODE
OUT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
ASPECTS = ["physical", "sensory", "language", "financial", "time"]
STATES = ["no_mention", "barrier_reported", "positive_access", "unclear_or_mixed"]
SERVICES = ["clinic", "pharmacy", "grocery"]
rng = np.random.default_rng(SEED)
PACKAGES = ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "folium", "ipywidgets", "datasketch", "joblib"]
environment = {p: version(p) for p in PACKAGES}
environment.update(python=sys.version, platform=platform.platform(), seed=SEED, mode=MODE)
(OUT / "environment.json").write_text(json.dumps(environment, indent=2))
print("SYNTHETIC SOFTWARE TEST — NOT REAL FINDINGS" if MODE == "synthetic" else "REAL DATA — inspect label provenance before interpreting outputs")


## 2 · Data acquisition and fixtures

Use the full `google_maps.csv` and `google_maps_reviews.csv`, not the samples, from the [organizer folder](https://drive.google.com/drive/u/1/folders/1l4JdVQOm5iaS0WO0tQ1PL-fCtqWT_8aj). Access and licensing must be checked by your team. Upload via Colab's Files panel into `/content/data`, or set `DATA_DIR` to a mounted Drive folder. This notebook does not automatically access an account or download third-party data.

Required columns: Places `place_id, name, category, lat, lon`; Reviews `place_id, review`. Other columns are optional and audited. Reviewer identity and profile URLs are discarded immediately. Source checksums identify exact input versions. Synthetic place names, coordinates, text, and labels below are invented solely for testing. Repeated language templates make their metrics unrealistically easy.


In [ ]:
def digest(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()

fixtures = {
 "physical": ["The entrance has steps and no ramp.", "The ramp provided easy wheelchair access.", "There is a ramp but it was sometimes blocked."],
 "sensory": ["No hearing loop was available for communication.", "The hearing loop worked well.", "The hearing loop worked intermittently."],
 "language": ["No interpreter was available when requested.", "An interpreter helped me receive the service.", "An interpreter was offered but assistance was inconsistent."],
 "financial": ["I could not afford the fee so I left without service.", "Financial assistance covered the fee.", "The fees were waived once but charged later."],
 "time": ["I waited three hours and could not get an appointment.", "I got an appointment promptly without waiting.", "Some appointments were prompt but others had long waits."]
}
synthetic_truth = {}
source_manifest = {}
if MODE == "synthetic":
    ps, rs = [], []
    # Distinct context prevents identical records; shared templates remain a disclosed limitation.
    contexts = ["morning", "afternoon", "evening", "weekday", "weekend", "holiday"]
    for i in range(180):
        pid = f"fixture_{i:03d}"
        ps.append(dict(place_id=pid, name=f"SYNTHETIC Service {i}", category=SERVICES[i % 3],
                       lat=37.7 + rng.uniform(-.12, .12), lon=-122.3 + rng.uniform(-.12, .12),
                       open_hours="unknown", reviews_count=999))
        for j in range(5):
            states = {a: "no_mention" for a in ASPECTS}
            selected = rng.choice(ASPECTS, size=int(rng.integers(0, 3)), replace=False)
            sentences = []
            for a in selected:
                k = int(rng.integers(3)); states[a] = STATES[k+1]; sentences.append(fixtures[a][k])
            if not sentences: sentences = ["The staff were friendly and the decor was pleasant."]
            text = " ".join(sentences) + f" Visit {i}-{j}, {contexts[(i+j)%6]}."
            rs.append(dict(place_id=pid, review=text, review_date=f"2026-{1+i%10:02d}-{1+j:02d}",
                           response_of_owner="Thank you for your feedback." if j == 0 else ""))
            synthetic_truth[digest(pid + "|" + text)[:24]] = states
    places_raw, reviews_raw = pd.DataFrame(ps), pd.DataFrame(rs)
    source_manifest = {"source": "generated synthetic fixture", "places": len(ps), "reviews": len(rs)}
elif MODE == "real":
    pp, rp = DATA_DIR / "google_maps.csv", DATA_DIR / "google_maps_reviews.csv"
    missing = [str(p) for p in [pp, rp] if not p.exists()]
    if missing:
        raise FileNotFoundError("Upload the FULL organizer CSVs using the Files panel: " + ", ".join(missing))
    places_raw = pd.read_csv(pp, dtype={"place_id": "string"}, low_memory=False)
    reviews_raw = pd.read_csv(rp, dtype={"place_id": "string"}, low_memory=False)
    source_manifest = {p.name: {"sha256": file_sha(p), "bytes": p.stat().st_size} for p in [pp, rp]}
else:
    raise ValueError("MODE must be synthetic or real")
(OUT / "sources.json").write_text(json.dumps(source_manifest, indent=2))
print({"mode": MODE, "places_rows": len(places_raw), "review_rows": len(reviews_raw)})


## 3 · Audit, canonical Places records, and a validated join

Duplicate place IDs are resolved by highest field completeness, with deterministic source-order tie breaking. Categories from duplicate records are unioned without multiplying reviews. Categories and coordinates always come from Places. We reject blank IDs; missing coordinates are retained for lists, excluded from maps. Unknown hours and closure flags stay unknown.

Inspect the displayed audit and random sample before selecting scope. The geographic description is **locations in the supplied dataset**, not a claim that every point is within the nine Bay Area counties. No county boundary filter is applied. Keyword hit rates below are a screening statistic, not human-verified barrier prevalence.


In [ ]:
required = [(places_raw, {"place_id", "name", "category", "lat", "lon"}), (reviews_raw, {"place_id", "review"})]
for frame, columns in required:
    assert columns <= set(frame.columns), f"Missing columns: {columns - set(frame.columns)}; found {list(frame.columns)}"
audit = {}
for name, frame in [("places", places_raw), ("reviews", reviews_raw)]:
    audit[name] = {"rows": len(frame), "columns": list(frame.columns),
                   "unique_place_ids": int(frame.place_id.nunique()),
                   "duplicate_rows": int(frame.duplicated().sum()),
                   "missing_fraction": frame.isna().mean().round(4).to_dict()}
    frame["place_id"] = frame.place_id.astype("string").str.strip().replace("", pd.NA)

pcols = [c for c in ["place_id", "name", "category", "all_categories", "lat", "lon", "open_hours", "reviews_count"] if c in places_raw]
p = places_raw[pcols].dropna(subset=["place_id"]).copy()
p["_complete"] = p.notna().sum(axis=1)
category_columns = [c for c in ["category", "all_categories"] if c in p]
p["category_text"] = p[category_columns].fillna("").astype(str).agg(" | ".join, axis=1)
category_union = p.groupby("place_id").category_text.agg(lambda s: " | ".join(sorted(set(s))))
audit["places"]["duplicate_place_ids"] = int(p.place_id.duplicated().sum())
p = p.sort_values("_complete", ascending=False, kind="stable").drop_duplicates("place_id").drop(columns=["_complete", "category_text"])
p = p.join(category_union, on="place_id")
for c in ["lat", "lon"]: p[c] = pd.to_numeric(p[c], errors="coerce")
p["valid_coordinates"] = p.lat.between(-90,90) & p.lon.between(-180,180) & ~((p.lat == 0) & (p.lon == 0))
audit["invalid_coordinates"] = int((~p.valid_coordinates).sum())
rcols = [c for c in ["place_id", "review", "review_date", "response_of_owner"] if c in reviews_raw]
r = reviews_raw[rcols].dropna(subset=["place_id", "review"]).copy()
r["review"] = r.review.astype(str).str.strip()
r = r[r.review.str.len().gt(0)].copy()
for c in ["review_date", "response_of_owner"]:
    if c not in r: r[c] = ""
r["date"] = pd.to_datetime(r.review_date, errors="coerce", utc=True, format="mixed")
r["record_id"] = [digest(str(pid)+"|"+text)[:24] for pid,text in zip(r.place_id,r.review)]
joined = r.merge(p, on="place_id", how="left", validate="many_to_one", indicator=True)
audit["join"] = {"eligible_review_rows":len(r), "matched":int(joined._merge.eq("both").sum()),
                  "coverage":float(joined._merge.eq("both").mean()),
                  "unmatched_example_ids": joined.loc[joined._merge.eq("left_only"),"place_id"].head(10).tolist()}
audit["dates"] = {"min":str(r.date.min()), "max":str(r.date.max()), "unknown_fraction":float(r.date.isna().mean())}
audit["category_counts"] = p.category.fillna("unknown").value_counts().head(30).to_dict()
joined = joined[joined._merge.eq("both")].drop(columns="_merge").reset_index(drop=True)
assert len(joined) <= len(r) and p.place_id.is_unique
# Remove identity-bearing raw tables from the active analysis namespace.
del places_raw, reviews_raw
display(pd.DataFrame([{"dataset":k, "rows":audit[k]["rows"], "unique_places":audit[k]["unique_place_ids"]} for k in ["places","reviews"]]))
print(json.dumps({k:v for k,v in audit.items() if k not in ["places","reviews","category_counts"]}, indent=2))
display(joined[["record_id","category","review"]].sample(min(12,len(joined)), random_state=SEED))


## 4 · Scope, duplicates, and frozen place-held-out splits

Service assignment uses a documented precedence (pharmacy → grocery → clinic) and keeps one primary category per place. Inspect ambiguous categories; do not silently broaden scope when counts are small. Exact normalized duplicates are removed globally; approximate token-trigram duplicates use MinHash candidates followed by Jaccard ≥ 0.85. This approximate procedure can miss paraphrases; duplicate leakage checks are not proof against semantic leakage. Review counts after deduplication are **eligible sampled counts**, not lifetime Google counts.

Fixed place-group splits are approximately 70/15/15 by places, not necessarily reviews. Test labels are not loaded for fitting or model selection. Re-running against different input files creates different manifests; freeze and archive the current files before annotation.


In [ ]:
def service_group(s):
    s = str(s).lower()
    if re.search(r"pharmac|drugstore", s): return "pharmacy"
    if re.search(r"grocery|supermarket|food market", s): return "grocery"
    if re.search(r"clinic|medical|hospital|health center|doctor|dentist", s): return "clinic"
    return "other"
joined["service"] = joined.category_text.map(service_group)
audit["service_review_counts"] = joined.service.value_counts().to_dict()
df = joined[joined.service.isin(SERVICES)].copy()
assert len(df), "No matched target-service reviews. Inspect categories and document any scope change."
def normalize(s): return re.sub(r"[^\w\s]", " ", s.casefold()).strip()
df["norm"] = df.review.map(lambda s: " ".join(normalize(s).split()))
n_before = len(df)
df = df.sort_values("record_id").drop_duplicates("norm").copy()
audit["exact_duplicates_removed"] = n_before - len(df)
def shingles(s):
    words = s.split()
    return {" ".join(words[i:i+3]) for i in range(max(1,len(words)-2))}
lsh, signatures, retained = MinHashLSH(threshold=.80,num_perm=64), {}, []
for idx, text in zip(df.index, df.norm):
    tokens = shingles(text); mh = MinHash(num_perm=64, seed=SEED)
    for token in sorted(tokens): mh.update(token.encode())
    candidates = lsh.query(mh)
    duplicate = any(len(tokens & signatures[k])/max(1,len(tokens | signatures[k])) >= .85 for k in candidates)
    if not duplicate:
        key = str(idx); lsh.insert(key,mh); signatures[key]=tokens; retained.append(idx)
audit["near_duplicates_removed"] = len(df)-len(retained)
df = df.loc[retained].reset_index(drop=True)
assert df.place_id.nunique() >= 10, "Too few distinct places for defensible three-way splits; acquire more data or revise scope."
train_idx, rest_idx = next(GroupShuffleSplit(n_splits=1, test_size=.30, random_state=SEED).split(df, groups=df.place_id))
val_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=.5, random_state=SEED+1).split(df.iloc[rest_idx], groups=df.iloc[rest_idx].place_id))
df["split"] = "train"
df.loc[rest_idx[val_pos],"split"] = "validation"
df.loc[rest_idx[test_pos],"split"] = "test"
df[["record_id","place_id","split"]].to_csv(OUT/"split_manifest.csv",index=False)
display(pd.crosstab(df.split, df.service))
print("Retained", len(df), "reviews across", df.place_id.nunique(), "places")


## 5 · Annotation protocol and provisional labels

For **each review and each of the five aspects**, assign exactly one state:

| State | Meaning | Example |
|---|---|---|
| `no_mention` | No evidence about this aspect | “Friendly staff” does not establish access |
| `barrier_reported` | Reviewer reports an obstacle to receiving service | “Steps and no ramp” |
| `positive_access` | Explicit enabling accommodation | “Interpreter helped me” |
| `unclear_or_mixed` | Ambiguous, historical, hypothetical, or conflicting experience | “Ramp was sometimes blocked” |

Physical access is the flagship. Sensory includes hearing/signage/communication accommodation; language requires assistance in a requested language; financial requires an access-related cost obstacle (generic “expensive” is insufficient); time concerns appointments, waits, hours, or closures.

Copy an **exact evidence span** for every mention. Do not infer reviewer disability or demographics. Generic rudeness is not an access barrier. Positive ramps are not barriers. An owner saying “fixed” is only an owner claim. No mention is never positive evidence.

The queue includes a random review sample from each split; training also gets keyword-enriched examples. Label all five aspects for a sampled review. Reserve test rows for independent annotators; model developers must not read their labels until selection is frozen. Two people independently annotate an overlapping subset (e.g., 20%); use `annotations_double.csv` for agreement, adjudicate disagreements, then save final rows to `labels.csv` with `provenance=human`, `adjudicated=True`. Keep the original two annotations.

`labels.csv` fields: `record_id, aspect, state, evidence, annotator_id, provenance, adjudicated`. A review missing any aspect is excluded from gold modeling/evaluation. Weak labels below are intentionally conservative heuristics and **not human truth**. In real mode without enough gold train/validation labels, they only enable a provisional demo. A high match to the same labeling rules would be circular, so weak test scores are not reported.


In [ ]:
TERMS = {
 "physical": r"\b(wheelchair|ramp|stairs?|steps|elevator|accessible entrance|accessible restroom)\b",
 "sensory": r"\b(hearing loop|hearing aid|braille|sign language|accessible signage)\b",
 "language": r"\b(interpreter|translation|language assistance|spanish|mandarin)\b",
 "financial": r"\b(afford|financial assistance|unexpected fees?|could not pay|couldn't pay|fee waiver)\b",
 "time": r"\b(waited|long wait|appointment|unexpectedly closed|limited hours|closed early)\b"
}
NEG = r"\b(no|not|cannot|could not|couldn't|unavailable|broken|blocked|inaccessible|long|hours|unexpected|limited|unaffordable)\b"
POS = r"\b(easy|helped|available|accessible|worked|promptly|covered|waived|without waiting)\b"
MIX = r"\b(but|sometimes|intermittently|inconsistent|used to|might|would|if only)\b"
def rule_state(text, aspect):
    sentences = re.split(r"(?<=[.!?])\s+", str(text))
    relevant = [s for s in sentences if re.search(TERMS[aspect],s,re.I)]
    if not relevant: return "no_mention"
    s = " ".join(relevant).lower()
    if re.search(MIX,s): return "unclear_or_mixed"
    if re.search(r"\b(no problems?|not difficult|without waiting|no steps)\b",s): return "positive_access"
    if re.search(NEG,s): return "barrier_reported"
    if re.search(POS,s): return "positive_access"
    return "unclear_or_mixed"

df["screen_hit"] = df.review.map(lambda t:any(re.search(p,t,re.I) for p in TERMS.values()))
audit["screening_hit_fraction_NOT_prevalence"] = float(df.screen_hit.mean())
audit["retained_reviews"] = len(df)
(OUT/"data_audit.json").write_text(json.dumps(audit,indent=2,default=str))
queues=[]
for split, n in [("train",350),("validation",150),("test",150)]:
    pool = df[df.split.eq(split)]
    q = pool.sample(min(n,len(pool)), random_state=SEED).copy(); q["sampling"]="random"
    queues.append(q)
enriched = df[df.split.eq("train") & df.screen_hit].sample(min(100,int((df.split.eq("train") & df.screen_hit).sum())),random_state=SEED)
enriched=enriched.copy(); enriched["sampling"]="enriched_training"
queue=pd.concat(queues+[enriched]).drop_duplicates("record_id")
queue=queue[["record_id","review","service","split","sampling"]].merge(pd.DataFrame({"aspect":ASPECTS}),how="cross")
for c in ["state","evidence","annotator_id","provenance","adjudicated"]: queue[c]=""
queue.to_csv(OUT/"annotation_queue_PRIVATE.csv",index=False)

def load_gold(allowed_splits):
    if MODE == "synthetic":
        return pd.DataFrame([dict(record_id=r.record_id, **synthetic_truth[r.record_id]) for r in df.itertuples() if r.split in allowed_splits])
    path = DATA_DIR/"labels.csv"
    if not path.exists(): return pd.DataFrame(columns=["record_id"]+ASPECTS)
    # Filter partitions before validating/using labels; test never enters fitting.
    valid_ids=set(df.loc[df.split.isin(allowed_splits),"record_id"])
    chunks=[]
    for chunk in pd.read_csv(path,keep_default_na=False,chunksize=1000):
        chunks.append(chunk[chunk.record_id.isin(valid_ids)])
    labels=pd.concat(chunks,ignore_index=True)
    required={"record_id","aspect","state","evidence","annotator_id","provenance","adjudicated"}
    assert required <= set(labels), f"labels.csv needs {required}"
    labels=labels[labels.provenance.eq("human") & labels.adjudicated.astype(str).str.lower().isin(["true","1","yes"])]
    assert not labels.duplicated(["record_id","aspect"]).any(), "Adjudicate duplicate final labels first"
    assert labels.aspect.isin(ASPECTS).all() and labels.state.isin(STATES).all()
    lookup=df.set_index("record_id").review.to_dict()
    for row in labels.itertuples():
        assert row.annotator_id.strip(), "Missing annotator ID"
        assert row.state == "no_mention" or (row.evidence and row.evidence in lookup[row.record_id]), "Evidence must be an exact nonempty substring"
    if labels.empty: return pd.DataFrame(columns=["record_id"]+ASPECTS)
    return labels.pivot(index="record_id",columns="aspect",values="state").reindex(columns=ASPECTS).dropna().reset_index()

gold_dev=load_gold(["train","validation"])
dev=df[df.split.ne("test")].copy()
dev_gold=dev.merge(gold_dev,on="record_id",validate="one_to_one")
GOLD_TRAINING=all((dev_gold.split.eq(s).sum()>=10) for s in ["train","validation"])
if GOLD_TRAINING:
    dev=dev_gold
    LABEL_MODE="synthetic_known_states" if MODE=="synthetic" else "adjudicated_human"
else:
    for a in ASPECTS: dev[a]=dev.review.map(lambda t:rule_state(t,a))
    LABEL_MODE="provisional_rule_labels"
print("Training provenance:",LABEL_MODE,"| completed development gold reviews:",len(dev_gold))
# Agreement is calculated from original, independent judgments, not adjudicated copies.
double_path=DATA_DIR/"annotations_double.csv"
agreement=[]
if MODE=="real" and double_path.exists():
    double=pd.read_csv(double_path).query("record_id in @dev.record_id")
    assert not double.duplicated(["record_id","aspect","annotator_id"]).any()
    for a,g in double.groupby("aspect"):
        wide=g.pivot(index="record_id",columns="annotator_id",values="state")
        if wide.shape[1]==2:
            both=wide.dropna()
            agreement.append({"aspect":a,"overlap":len(both),"agreement":float((both.iloc[:,0]==both.iloc[:,1]).mean()),
                              "cohen_kappa":float(cohen_kappa_score(both.iloc[:,0],both.iloc[:,1])) if len(both)>1 else None})
display(pd.DataFrame(agreement) if agreement else Markdown("**Independent annotator agreement: pending.**"))


## 6 · Train models and freeze validation decisions

Compare a keyword baseline, trained word TF-IDF logistic regression, and a word+character TF-IDF variant. The second feature representation tests robustness to spelling/phrase variations without downloading a language model. Each aspect gets a four-state classifier (including no mention). A missing training state cannot be learned; constant-label aspects are handled explicitly. Features use review text only; Places provide scope, spatial context, and subgroup analysis, not hidden rating labels.

Each trained variant uses validation-only per-aspect barrier thresholds, then selects by validation barrier macro-F1 (ties retain the earlier baseline). Non-barrier predictions use the highest-probability alternative state. State probabilities are **uncalibrated model scores**, not verified confidence. Small validation sets and unsupported classes limit selection. No claim of model improvement is made unless actual held-out evidence supports it.


In [ ]:
train=dev[dev.split.eq("train")].reset_index(drop=True)
val=dev[dev.split.eq("validation")].reset_index(drop=True)
assert len(train) and len(val)
class AspectModel:
    def __init__(self, variant="word", C=1.0):
        self.variant,self.C=variant,C
        word=TfidfVectorizer(ngram_range=(1,2),min_df=1,max_df=1.0,max_features=40000,sublinear_tf=True,strip_accents="unicode")
        self.vectorizer=word if variant=="word" else FeatureUnion([("word",word),("char",TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,max_features=40000,sublinear_tf=True))])
    def fit(self, frame):
        X=self.vectorizer.fit_transform(frame.review)
        self.models={}
        for a in ASPECTS:
            y=frame[a].to_numpy()
            if len(set(y))<2: self.models[a]=str(y[0])
            else: self.models[a]=LogisticRegression(C=self.C,class_weight="balanced",max_iter=1000,random_state=SEED).fit(X,y)
        return self
    def predict_proba(self,texts):
        X=self.vectorizer.transform(list(texts)); output=np.zeros((X.shape[0],len(ASPECTS),len(STATES)))
        for j,a in enumerate(ASPECTS):
            model=self.models[a]
            if isinstance(model,str): output[:,j,STATES.index(model)]=1
            else:
                probs=model.predict_proba(X)
                for k,state in enumerate(model.classes_): output[:,j,STATES.index(state)]=probs[:,k]
        return output

def rule_probs(texts):
    return np.array([[[float(rule_state(t,a)==s) for s in STATES] for a in ASPECTS] for t in texts])
def decode(probs, thresholds):
    alternative=probs.copy(); alternative[:,:,1]=-1
    states=np.array(STATES)[alternative.argmax(axis=2)]
    states[probs[:,:,1]>=np.asarray(thresholds)[None,:]]="barrier_reported"
    return states
def barrier_matrix(frame): return frame[ASPECTS].eq("barrier_reported").to_numpy().astype(int)
def tune_thresholds(probs,y):
    thresholds=[]
    for j in range(len(ASPECTS)):
        if not y[:,j].any(): thresholds.append(1.01); continue
        candidates=np.arange(.2,.81,.05)
        scores=[f1_score(y[:,j],probs[:,j,1]>=t,zero_division=0) for t in candidates]
        thresholds.append(float(candidates[int(np.argmax(scores))]))
    return thresholds

models={"keywords":None}; thresholds={"keywords":[.5]*5}; validation=[]
yval=barrier_matrix(val)
for name,variant in [("keywords",None),("tfidf_word","word"),("tfidf_word_char","word_char")]:
    if variant:
        models[name]=AspectModel(variant).fit(train)
        probs=models[name].predict_proba(val.review)
        thresholds[name]=tune_thresholds(probs,yval)
    else: probs=rule_probs(val.review)
    pred=decode(probs,thresholds[name])
    validation.append({"model":name,"barrier_macro_f1":float(f1_score(yval,pred=="barrier_reported",average="macro",zero_division=0)),
                       "state_macro_f1":float(f1_score(val[ASPECTS].to_numpy().ravel(),pred.ravel(),labels=STATES,average="macro",zero_division=0)),"label_mode":LABEL_MODE})
validation=pd.DataFrame(validation)
winner=validation.sort_values("barrier_macro_f1",ascending=False,kind="stable").iloc[0].model
selection={"winner":winner,"thresholds":thresholds,"label_mode":LABEL_MODE,"validation":validation.to_dict("records"),"seed":SEED}
(OUT/"selection_FROZEN.json").write_text(json.dumps(selection,indent=2))
serialized={name: None if model is None else {"vectorizer":model.vectorizer,"classifiers":model.models,"variant":model.variant} for name,model in models.items()}
joblib.dump({"models":serialized,"thresholds":thresholds,"winner":winner,"aspects":ASPECTS,"states":STATES},OUT/"models_PRIVATE.joblib")
display(validation)
print("Frozen selected model:",winner)
print("Training state support:"); display(pd.DataFrame({a:train[a].value_counts() for a in ASPECTS}).fillna(0).astype(int))


## 7 · Held-out evaluation and place-cluster uncertainty

Only now load test labels. Report all predeclared candidates, but **do not change the frozen winner using test results**. In real mode, only adjudicated human test labels support these metrics. If annotations are incomplete, results describe the labeled subset; they are not representative prevalence estimates. Three service subgroups are predeclared: clinics, pharmacies, groceries. Missing groups remain unsupported; do not invent substitute groups after seeing test performance.

Metrics include aspect precision/recall/F1 and support, micro/macro-F1, four-state confusion matrices, top-20 barrier precision, threshold/coverage tradeoffs, and cluster-bootstrap intervals. Classes without test positives are marked unsupported. Macro-F1 over all five aspects is shown for comparability; also report supported-aspect macro-F1. With few places, bootstrap intervals can be unstable and are not evidence of fairness across disability communities.


In [ ]:
gold_test=load_gold(["test"])
test=df[df.split.eq("test")].merge(gold_test,on="record_id",validate="one_to_one").reset_index(drop=True)
EVALUATION_STATUS=("SYNTHETIC TEST ONLY" if MODE=="synthetic" else "HUMAN-LABELED SUBSET; check sampling") if len(test) else "BLOCKED: independent test labels unavailable"
metrics={"status":EVALUATION_STATUS,"n_labeled_test":len(test),"n_total_test":int(df.split.eq("test").sum()),"models":{}}
per_aspect_rows=[]; subgroup_rows=[]; tradeoffs=[]; test_predictions={}; test_probabilities={}
def summarize(y,pred):
    pr,rc,f,s=precision_recall_fscore_support(y,pred,average=None,zero_division=0)
    supported=s>0
    return {"micro_f1":float(f1_score(y,pred,average="micro",zero_division=0)),
            "macro_f1_all_aspects":float(f.mean()),"macro_f1_supported":float(f[supported].mean()) if supported.any() else None}, (pr,rc,f,s)
if len(test):
    yt=barrier_matrix(test)
    for name,model in models.items():
        prob=rule_probs(test.review) if model is None else model.predict_proba(test.review)
        pred=decode(prob,thresholds[name]); yp=(pred=="barrier_reported").astype(int)
        test_predictions[name]=pred; test_probabilities[name]=prob
        summary,(pr,rc,f,s)=summarize(yt,yp)
        summary["state_macro_f1"]=float(f1_score(test[ASPECTS].to_numpy().ravel(),pred.ravel(),labels=STATES,average="macro",zero_division=0))
        summary["state_confusion_matrix"]=confusion_matrix(test[ASPECTS].to_numpy().ravel(),pred.ravel(),labels=STATES).tolist()
        summary["per_aspect_state_confusion"]={a:confusion_matrix(test[a],pred[:,j],labels=STATES).tolist() for j,a in enumerate(ASPECTS)}
        for j,a in enumerate(ASPECTS):
            flagged=np.flatnonzero(yp[:,j]); top=flagged[np.argsort(-prob[flagged,j,1],kind="stable")[:20]]
            per_aspect_rows.append(dict(model=name,aspect=a,precision=float(pr[j]),recall=float(rc[j]),f1=float(f[j]),positive_support=int(s[j]),
                                        predicted_positive=int(yp[:,j].sum()),supported=bool(s[j]>0),top_k=len(top),top_k_precision=float(yt[top,j].mean()) if len(top) else None))
        metrics["models"][name]=summary
    pred=test_predictions[winner]; yp=(pred=="barrier_reported").astype(int); prob=test_probabilities[winner]
    for service in SERVICES:
        mask=test.service.eq(service).to_numpy()
        result={"service":service,"reviews":int(mask.sum()),"places":int(test.loc[mask,"place_id"].nunique()),"positive_labels":int(yt[mask].sum()),"status":"supported" if mask.sum() and yt[mask].sum() else "insufficient support"}
        if mask.any(): result.update(summarize(yt[mask],yp[mask])[0])
        subgroup_rows.append(result)
    clusters={pid:np.flatnonzero(test.place_id.to_numpy()==pid) for pid in test.place_id.unique()}
    boot=[]; brng=np.random.default_rng(SEED)
    if len(clusters)>=5:
        ids=list(clusters)
        for _ in range(200):
            ix=np.concatenate([clusters[k] for k in brng.choice(ids,size=len(ids),replace=True)])
            boot.append(f1_score(yt[ix],yp[ix],average="micro",zero_division=0))
    metrics["winner_micro_f1_cluster_CI95"]=np.quantile(boot,[.025,.975]).tolist() if boot else None
    metrics["bootstrap_places"]=len(clusters)
    # Descriptive test tradeoff curves only; never use them to retune the winner.
    for t in [.3,.5,.7,.9]:
        flags=prob[:,:,1]>=t
        tradeoffs.append({"threshold":t,"flagged_pairs":int(flags.sum()),"pair_coverage":float(flags.mean()),"precision":float(yt[flags].mean()) if flags.any() else None})
    display(pd.DataFrame(per_aspect_rows)); display(pd.DataFrame(subgroup_rows)); display(pd.DataFrame(tradeoffs))
    cm=np.array(metrics["models"][winner]["state_confusion_matrix"])
    fig,ax=plt.subplots(figsize=(7,5)); ax.imshow(cm,cmap="Blues")
    ax.set(xticks=range(4),yticks=range(4),xticklabels=STATES,yticklabels=STATES,xlabel="Predicted",ylabel="Labeled",title=EVALUATION_STATUS)
    plt.setp(ax.get_xticklabels(),rotation=30,ha="right")
    for i in range(4):
        for j in range(4): ax.text(j,i,str(cm[i,j]),ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
    fig.tight_layout(); fig.savefig(OUT/"state_confusion.png",dpi=150); plt.show()
else:
    display(Markdown("**Independent evaluation is blocked.** Complete the reserved test annotation queue; no weak-label test metrics are substituted."))
metrics["subgroups"]=subgroup_rows
(OUT/"metrics.json").write_text(json.dumps(metrics,indent=2,allow_nan=False))
pd.DataFrame(per_aspect_rows).to_csv(OUT/"per_aspect_metrics.csv",index=False)
pd.DataFrame(tradeoffs).to_csv(OUT/"threshold_tradeoffs.csv",index=False)


## 8 · Error analysis and annotation review

Examine false positives/negatives and four-state errors, including positive accommodation, negation, mixed language, short text, and rare terminology. The slices below overlap and are diagnostic—not demographic groups. Non-ASCII text is only a **language-review queue**, never an inferred language label. Have a qualified speaker identify languages actually present before claiming multilingual performance. Human conclusions remain pending until someone examines the examples. Do not edit the model using this held-out error analysis; use a newly held-out set for the next model version.


In [ ]:
errors=[]; slice_rows=[]
if len(test):
    pred=test_predictions[winner]
    for i,row in enumerate(test.itertuples()):
        for j,a in enumerate(ASPECTS):
            true=getattr(row,a)
            if true!=pred[i,j]:
                errors.append({"record_id":row.record_id,"service":row.service,"aspect":a,"truth":true,"predicted":pred[i,j],"review_PRIVATE":row.review})
    slices={"negation":test.review.str.contains(r"\b(no|not|without|never)\b",case=False,regex=True),
            "positive_accommodation":test[ASPECTS].eq("positive_access").any(axis=1),
            "mixed_experience":test[ASPECTS].eq("unclear_or_mixed").any(axis=1),
            "short_under_8_words":test.review.str.split().str.len()<8,
            "non_ASCII_review_needed":test.review.map(lambda s:any(ord(c)>127 for c in s)),
            "rare_terms":test.review.str.contains(r"braille|hearing loop|sign language",case=False,regex=True)}
    for name,mask in slices.items():
        slice_rows.append({"slice":name,"n":int(mask.sum()),"state_accuracy":float((test.loc[mask,ASPECTS].to_numpy()==pred[mask]).mean()) if mask.any() else None})
display(pd.DataFrame(slice_rows))
display(pd.DataFrame(errors).head(20))
pd.DataFrame(errors).to_csv(OUT/"error_review_PRIVATE.csv",index=False)
pd.DataFrame(slice_rows).to_csv(OUT/"error_slices.csv",index=False)


## 9 · Evidence extraction and transparent place aggregation

Inference covers all deduplicated eligible reviews; only test-partition gold labels support evaluation. The map is a provisional demonstration when training/evaluation are provisional. Evidence is an exact relevant sentence, or the entire review if no sentence matches; extraction accuracy is not evaluated. Long text is not silently turned into an invented quotation. Owner responses are retained locally as unverified claims, never classified as reviewer evidence.

Aggregate separately by place and aspect. Conflicting evidence means positive and barrier reports coexist, or mixed/unclear reports occur. No evidence means insufficient evidence, not an accessible place. Scores are uncalibrated; sorting uses distinct reports, recency, or score without a universal accessibility/severity score. Population prevalence and current conditions cannot be inferred from a biased review sample.


In [ ]:
chosen=models[winner]
probs=rule_probs(df.review) if chosen is None else chosen.predict_proba(df.review)
states=decode(probs,thresholds[winner])
def evidence_for(text,aspect):
    sentences=re.split(r"(?<=[.!?])\s+",text)
    return next((s for s in sentences if re.search(TERMS[aspect],s,re.I)),text)
rows=[]
for i,row in enumerate(df.itertuples()):
    for j,a in enumerate(ASPECTS):
        state=states[i,j]; evidence="" if state=="no_mention" else evidence_for(row.review,a)
        rows.append(dict(record_id=row.record_id,place_id=row.place_id,name=row.name,service=row.service,aspect=a,state=state,
                         score=float(probs[i,j,STATES.index(state)]),barrier_score=float(probs[i,j,1]),date=row.date,
                         lat=row.lat,lon=row.lon,valid_coordinates=row.valid_coordinates,evidence=evidence,
                         owner_claim=row.response_of_owner,provenance=LABEL_MODE,model=winner))
evidence=pd.DataFrame(rows)
def aggregate(frame):
    out=[]
    for (pid,a),g in frame.groupby(["place_id","aspect"],sort=True):
        counts=g.state.value_counts(); barrier=int(counts.get("barrier_reported",0)); positive=int(counts.get("positive_access",0)); uncertain=int(counts.get("unclear_or_mixed",0))
        status="conflicting_or_unclear" if (barrier and positive) or uncertain else "reported_barrier" if barrier else "positive_evidence" if positive else "insufficient_evidence"
        meta=g.iloc[0]
        out.append(dict(place_id=pid,aspect=a,name=meta["name"],service=meta.service,lat=meta.lat,lon=meta.lon,valid_coordinates=meta.valid_coordinates,
                        sampled_reviews=int(g.record_id.nunique()),barrier_reports=barrier,positive_reports=positive,uncertain_reports=uncertain,
                        status=status,first_date=g.date.min(),last_date=g.date.max(),mean_score=float(g.score.mean()),
                        evidence_ids=g.loc[g.state.ne("no_mention"),"record_id"].tolist()))
    return pd.DataFrame(out)
place_aspects=aggregate(evidence)
def summary(row):
    return (f"{row['name']} — {row['aspect']}: {row['barrier_reports']} predicted barrier, "
            f"{row['positive_reports']} positive and {row['uncertain_reports']} uncertain reports among "
            f"{row['sampled_reviews']} eligible sampled reviews. Status: {row['status']}. "
            f"Evidence IDs: {', '.join(row['evidence_ids']) or 'none'}. Manual verification required.")
place_aspects["summary"]=place_aspects.apply(summary,axis=1)
questions={"physical":"Confirm whether entrances and relevant facilities offer step-free access.",
           "sensory":"Confirm which sensory and communication accommodations are available.",
           "language":"Confirm interpreter or language-assistance availability.",
           "financial":"Confirm fees and assistance options that affect access.",
           "time":"Confirm hours, wait times and appointment availability."}
place_aspects["verification_question"]=place_aspects.aspect.map(questions)
verification_queue=place_aspects[place_aspects.barrier_reports.gt(0)|place_aspects.uncertain_reports.gt(0)].sort_values(["barrier_reports","last_date"],ascending=False)
# No review quotations or names in the minimal export; IDs still require data-rights review.
export_cols=["place_id","aspect","service","sampled_reviews","barrier_reports","positive_reports","uncertain_reports","last_date","evidence_ids","verification_question"]
verification_queue[export_cols].to_csv(OUT/"verification_queue_PRIVATE.csv",index=False)
display(verification_queue[["name","aspect","sampled_reviews","barrier_reports","positive_reports","status"]].head(15))


## 10 · Interactive evidence map and keyboard-accessible list

The Colab dashboard below filters service, aspect, recency, and **model score**. Unknown dates can be retained explicitly. Filtering happens at review level, then counts are recomputed. Text labels and a table accompany colors. Place details show exact supporting excerpts locally and distinguish owner claims. Do not publish screenshots containing real review text or identity information without review. Folium map tiles request OpenStreetMap tiles in your browser; no review text is sent to an LLM.


In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
service_w=widgets.Dropdown(options=["all"]+SERVICES,description="Service")
aspect_w=widgets.Dropdown(options=ASPECTS,description="Aspect")
days_w=widgets.Dropdown(options=[("All dates",0),("Last 90 days",90),("Last year",365)],description="Recency")
score_w=widgets.FloatSlider(value=0,min=0,max=1,step=.05,description="Min score")
unknown_w=widgets.Checkbox(value=True,description="Include unknown dates")
sort_w=widgets.Dropdown(options=["barrier_reports","last_date","mean_score"],description="Sort")
place_w=widgets.Dropdown(options=[("Choose a place","")]+[(str(n),str(p)) for p,n in df[["place_id","name"]].drop_duplicates().itertuples(index=False,name=None)],description="Details")
output_w=widgets.Output()
colors={"reported_barrier":"red","positive_evidence":"green","conflicting_or_unclear":"orange","insufficient_evidence":"gray"}
def filter_evidence(service,aspect,days,score,unknown):
    e=evidence[evidence.aspect.eq(aspect) & evidence.score.ge(score)].copy()
    if service!="all": e=e[e.service.eq(service)]
    if days:
        cutoff=pd.Timestamp.now(tz="UTC")-pd.Timedelta(days=days)
        e=e[e.date.ge(cutoff)|(e.date.isna() & unknown)]
    elif not unknown: e=e[e.date.notna()]
    return e
def refresh(change=None):
    with output_w:
        clear_output(wait=True)
        display(Markdown(f"**{MODE.upper()} · {LABEL_MODE} · {EVALUATION_STATUS}**"))
        e=filter_evidence(service_w.value,aspect_w.value,days_w.value,score_w.value,unknown_w.value)
        if e.empty:
            print("No evidence matches these filters; this does not establish positive access."); return
        agg=aggregate(e).sort_values(sort_w.value,ascending=False,na_position="last")
        pts=agg[agg.valid_coordinates]
        center=[float(pts.lat.median()),float(pts.lon.median())] if len(pts) else [37.77,-122.3]
        m=folium.Map(location=center,zoom_start=10,tiles="OpenStreetMap")
        # Cap rendering for browser responsiveness; the full list/export remains available.
        for row in pts.head(500).to_dict("records"):
            tip=f"{row['name']} | {row['status']} | {row['barrier_reports']}/{row['sampled_reviews']} predicted barrier reports"
            folium.CircleMarker([row["lat"],row["lon"]],radius=6,color=colors[row["status"]],fill=True,tooltip=html.escape(tip),popup=html.escape(summary(row))).add_to(m)
        display(m)
        print(f"Map: {min(500,len(pts))} of {len(pts)} mappable places. Table: first 100 of {len(agg)}. Missing coordinates stay in list.")
        display(agg[["name","service","status","sampled_reviews","barrier_reports","positive_reports","uncertain_reports","last_date"]].head(100))
        agg["verification_question"]=agg.aspect.map(questions)
        agg[export_cols].to_csv(OUT/"filtered_queue_PRIVATE.csv",index=False)
        if place_w.value:
            detail=e[e.place_id.eq(place_w.value) & e.state.ne("no_mention")]
            display(Markdown("**Place evidence (selected filters)**"))
            if detail.empty: print("No matching reported evidence for this place.")
            for row in detail.head(20).to_dict("records"):
                display(HTML(f"<p><b>{html.escape(row['state'])}</b> · {html.escape(row['record_id'])} · {html.escape(str(row['date']))}<br>{html.escape(row['evidence'])}</p>"))
                if str(row["owner_claim"]).strip() and pd.notna(row["owner_claim"]):
                    display(HTML("<p><em>Unverified owner claim:</em> "+html.escape(str(row["owner_claim"]))+"</p>"))
        m.save(str(OUT/"map_PRIVATE.html"))
for w in [service_w,aspect_w,days_w,score_w,unknown_w,sort_w,place_w]: w.observe(refresh,names="value")
display(widgets.VBox([widgets.HBox([service_w,aspect_w,days_w]),widgets.HBox([score_w,unknown_w]),widgets.HBox([sort_w,place_w]),output_w]))
refresh()


## 11 · Integrity checks and summary audit

These assertions test join cardinality, split isolation, output shapes, evidence references, deterministic summary counts, and empty/unknown/conflicting states. They do not establish real-world model validity. The fixed synthetic fixture also tests the end-to-end pipeline. Manually review the summary audit sample before any presentation; extracted text can still contain personal information even after reviewer identity columns are removed.


In [ ]:
assert p.place_id.is_unique
assert df.record_id.is_unique and df.norm.is_unique
sets={s:set(df.loc[df.split.eq(s),"place_id"]) for s in ["train","validation","test"]}
assert not (sets["train"]&sets["test"] or sets["train"]&sets["validation"] or sets["test"]&sets["validation"])
assert probs.shape==(len(df),5,4) and np.allclose(probs.sum(axis=2),1)
assert set(evidence.state)<=set(STATES)
assert evidence.record_id.isin(df.record_id).all()
review_lookup=df.set_index("record_id").review.to_dict()
assert all(not r.evidence or r.evidence in review_lookup[r.record_id] for r in evidence.itertuples())
assert (place_aspects.barrier_reports+place_aspects.positive_reports+place_aspects.uncertain_reports<=place_aspects.sampled_reviews).all()
assert all(set(ids)<=set(df.record_id) for ids in place_aspects.evidence_ids)
assert filter_evidence("nonexistent",ASPECTS[0],0,0,True).empty
probe=evidence.iloc[:2].copy(); probe["place_id"]="test"; probe["aspect"]="physical"
probe["state"]=["barrier_reported","positive_access"]; probe["record_id"]=["a","b"]
assert aggregate(probe).iloc[0].status=="conflicting_or_unclear"
probe["state"]="no_mention"; probe["date"]=pd.NaT
assert aggregate(probe).iloc[0].status=="insufficient_evidence" and pd.isna(aggregate(probe).iloc[0].last_date)
summary_audit=place_aspects.sample(min(20,len(place_aspects)),random_state=SEED)[["place_id","aspect","summary"]].copy()
summary_audit["human_checked"]=""; summary_audit["notes"]=""
summary_audit.to_csv(OUT/"summary_audit_PRIVATE.csv",index=False)
print("PASS: join, split isolation, unique records, probability shape, evidence references, aggregation and empty/conflicting/unknown states.")


## 12 · Results report, model card, presentation, and timed video script

This cell generates a report from actual runtime outputs and a printable HTML slide deck. Synthetic runs are labeled on every slide. A 4:30 storyboard is supplied; no video file is fabricated. Real-data results, independent annotations/agreement, summary inspection, stakeholder validation, and the recording must be completed before claiming a finished competition submission.

**Interpretation:** Reviewers are self-selected, time-lagged, and unevenly distributed across services. Lack of a report does not establish access. Repeated reports justify checking conditions, not claiming causality or legal noncompliance. Service-group robustness does not measure fairness across disability communities. Joining Places improves category/spatial context; no predictive lift from Places is claimed.


In [ ]:
result_text=json.dumps(metrics,indent=2)
report=f"""# Access Beyond the Map — {MODE.upper()}

## Findings and status
{EVALUATION_STATUS}. Training provenance: {LABEL_MODE}. Frozen selected model: {winner}.
Loaded review rows: {audit['reviews']['rows']}; matched rows: {audit['join']['matched']}; join coverage: {audit['join']['coverage']:.2%}.
Retained deduplicated essential-service reviews: {len(df)}; places: {df.place_id.nunique()}.
These counts describe the current runtime inputs; synthetic mode does not describe actual businesses.

## Method
Canonical Places joined many-to-one to Reviews using place_id. Review identities discarded.
Global exact/approximate duplicate removal precedes place-held-out splits. Keyword baseline compared with trained word and word+character TF-IDF logistic regression. Validation-only thresholds and selection, then held-out evaluation.
Places contribute service categories and map coordinates. No star-based accessibility labels or claim of predictive uplift from geographic context.

## Evaluation
```json
{result_text}
```

## Limitations and remaining work
Synthetic scores only test software. Real runs with rule labels remain provisional.
Independent human labels, double annotation, summary/error inspection and real source audit are required for substantive claims.
Approximate deduplication may miss paraphrases. Chain leakage is not tested. No verified language-specific performance or disability-community fairness assessment.
Review selection bias, label scarcity, missing metadata and historical changes constrain interpretation.
No unsupported certifications, causal estimates or population prevalence claims.
Video recording and stakeholder validation remain outstanding. Do not submit automatically.
"""
(OUT/"final_report.md").write_text(report)
model_card=f"""# Model card
Purpose: prioritize manual verification of reported service-access barriers.
Version: source checksums + split_manifest.csv + selection_FROZEN.json + environment.json.
Inputs: review text; five aspect-specific four-state classifiers.
Model: {winner}; provenance: {LABEL_MODE}; evaluation: {EVALUATION_STATUS}.
Training partitions never include test places. Thresholds frozen before test evaluation.
Scores are uncalibrated. Unsupported aspects cannot be treated as validated.
Do not use for legal compliance, medical quality, enforcement, certification, or sensitive-trait inference.
Outputs concern sampled historical reports, not current verified conditions.
Raw data, annotations, quotes and model artifacts remain private pending rights review.
Only load joblib artifacts you generated or trust; pickle-based artifacts can execute code.
"""
(OUT/"model_card.md").write_text(model_card)
slide_content=[
 ("Access Beyond the Map","Maps locate services. We surface reported barriers for manual verification."),
 ("Two datasets, one evidence trail",f"Places categories and coordinates + Reviews text. {len(df)} retained reviews / {df.place_id.nunique()} places in this {MODE} run. Join coverage: {audit['join']['coverage']:.1%}."),
 ("Disability access first","Physical, sensory, language, financial and time aspects. Barrier, positive, mixed and no-mention states; exact evidence references."),
 ("Training and evaluation",f"Keyword baseline vs word and word+character TF-IDF classifiers. Place-held-out splits. Validation selects {winner}. {EVALUATION_STATUS}."),
 ("Evidence explorer","Map plus keyboard-accessible list, service/aspect/recency filters, conflicting evidence, private verification queue. No universal accessibility score."),
 ("What the results establish", "Synthetic results establish software behavior only." if MODE=="synthetic" else f"Test subset: {len(test)} labeled reviews. Consult metrics.json for support, subgroup results and uncertainty; avoid unsupported generalization."),
 ("Responsible next steps","Obtain and audit real data, annotate independently, inspect errors and summaries, verify on site, record a video under five minutes. No automatic accusations or outreach.")]
slides="<!doctype html><html lang='en'><meta charset='utf-8'><title>Access Beyond the Map</title><style>body{font-family:system-ui;background:#eef3f5;color:#123;margin:0}section{box-sizing:border-box;min-height:95vh;padding:9vw;page-break-after:always}h1{font-size:44px}p{font-size:26px;line-height:1.5}.status{font-size:18px;color:#744800}@media print{section{height:100vh}}</style>"
for title,body in slide_content:
    slides+=f"<section><div class='status'>{html.escape(MODE.upper()+' · '+EVALUATION_STATUS)}</div><h1>{html.escape(title)}</h1><p>{html.escape(body)}</p></section>"
slides+="</html>"
(OUT/"presentation.html").write_text(slides)
script="""# 4:30 video script/storyboard — RECORDING OUTSTANDING
0:00–0:30 — Slide 1: Explain how location availability differs from usable access. State this run's real/synthetic status explicitly.
0:30–1:05 — Slide 2: Show actual audit counts and the validated Places–Reviews join. Explain missingness and sampled-review coverage.
1:05–1:40 — Slide 3: Explain five aspects and four states using only permitted evidence. No mention is not positive access.
1:40–2:20 — Slide 4: Show trained classifiers, baseline comparison, place-held-out splits and label provenance. If labels are missing, say independent evaluation is blocked.
2:20–3:15 — Dashboard: Filter physical access and service type; inspect a reported barrier, conflicting report and unknown state. Show evidence IDs and verification questions.
3:15–3:55 — Slide 6: Present actual held-out metrics, positive support, three subgroups and uncertainty; in synthetic mode describe software checks only.
3:55–4:30 — Slide 7: Explain sampling bias, manual verification, next steps and what remains incomplete. Do not claim stakeholder validation or an impact estimate.
Export a recorded MP4 only after rights review. This script is not a completed video submission.
"""
(OUT/"video_script.md").write_text(script)
display(Markdown(report[:report.index("## Evaluation")]))
print("Generated final_report.md, model_card.md, presentation.html and video_script.md")


## 13 · Save reproducibility artifacts and download

The ZIP includes PRIVATE runtime artifacts (annotation queue, error examples, filtered map, trained models) and must not be posted publicly without review. The committed notebook intentionally has no outputs or datasets. Synthetic test files are reproducible from the seed; actual datasets require organizer permission.

For a real run: upload both full CSVs → set `MODE="real"` → audit and inspect samples → label/adjudicate the saved queues → place `labels.csv` in `DATA_DIR` → restart/run all with frozen source files. Preserve the initial split manifest and do not revise the model based on test errors. If reviewing labels changes the schema, create a new study version and hold-out set.

**Acceptance gates:** runnable implementation and integrity checks; real data audit; independent labels and agreement; trained-model comparison and three supported subgroups; evidence inspection; report/slides; actual video. The runtime checklist below distinguishes code completion from research completion.


**Authoring validation:** All Python analysis cells were executed sequentially in a local Python 3.12 process on the synthetic fixture; integrity assertions and file exports passed. The local environment used available package versions, recorded by the notebook, and skipped the Colab installation cell. Colab runtime execution and interactive browser usability have not been independently verified.


In [ ]:
checklist={"implementation_and_integrity_checks":True,"real_sources_loaded":MODE=="real",
           "human_development_labels_used":LABEL_MODE=="adjudicated_human",
           "independent_human_test_labels_available":MODE=="real" and len(test)>0,
           "three_supported_service_subgroups":MODE=="real" and len(subgroup_rows)==3 and all(r["status"]=="supported" for r in subgroup_rows),
           "independent_agreement_computed":bool(agreement),"human_summary_audit_complete":False,
           "report_and_slide_deck_generated":True,"actual_video_recorded":False,"stakeholder_validation":False}
(OUT/"acceptance_checklist.json").write_text(json.dumps(checklist,indent=2))
bundle=shutil.make_archive(str(OUT.parent/("access_beyond_map_"+MODE)),"zip",OUT)
display(pd.Series(checklist,name="complete"))
print("Saved private runtime bundle:",bundle)
# Explicit download control; no public hosting or GitHub upload occurs here.
download_button=widgets.Button(description="Download results ZIP",button_style="primary")
def download_results(_):
    try:
        from google.colab import files
        files.download(bundle)
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(bundle))
download_button.on_click(download_results)
display(download_button)
